# CGAN Comparativo e Treino Final - ArtBench-10

## Indice

- **1. Setup, dados e protocolo** - carregamento do ArtBench-10, transforms, seeds
- **2. Pipeline CGAN** - funcoes de treino, avaliacao e modelos (CGANGenerator, CGANDiscriminator)
- **3. Configuracao** - arquitetura condicional, adaptacao das melhorias do DCGAN e hiperparametros
- **4. Grid Search** - optimizacao de hiperparametros (subset 20%, 15 epocas)
- **5. Estudo de Ablacao** - V1 Vanilla, V2 TTUR, V3 TTUR+LS, V4 Full
- **6. Treino Final** - dataset completo, 400 epocas
- **7. Avaliacao Final** - 10 seeds, 5000 amostras, FID/KID rigoroso

### Resultados Finais
| Metrica | Valor |
|---------|-------|
| FID (5000 amostras, 10 seeds) | **99.65 +/- 1.26** |
| KID | **0.0744 +/- 0.0012** |
| Melhor Quick-FID (treino) | 111.82 |

## Setup do Ambiente Virtual e Dependencias

As dependencias base sao as mesmas usadas no resto do projeto:

```bash
pip install torch torchvision matplotlib datasets pillow "torchmetrics[image]" tqdm
pip install ipywidgets
```

Como este notebook depende de funcoes externas do ficheiro `cgan_maturity_study.py`, e importante correr tudo a partir da raiz do repositorio ou garantir que os caminhos locais para `scripts/` e `ArtBench-10/` estao corretos.


## 1. Setup, dados e protocolo

Mantemos a mesma disciplina experimental usada nas restantes familias:

1. `dev_loader` e `dev_loader_aug` para afinacao em subset.
2. `full_train_loader` e `full_train_loader_aug` para o treino final em 100% do treino.
3. Avaliacao final com numero fixo de amostras e multiplas repeticoes controladas por seed.

Esta separacao e especialmente importante no caso condicional, porque a tentacao de interpretar melhorias locais por classe como melhoria global do modelo pode ser enganadora. Um CGAN pode alinhar-se melhor com algumas labels e, mesmo assim, degradar a diversidade total ou introduzir artefactos estruturais que so aparecem nas metricas finais.


In [ ]:
# Inline plotting is enabled by the notebook frontend.

# ==========================================
# SETUP CONSOLIDADO E COMPLETO (CGAN)
# ==========================================
from __future__ import annotations
import sys
import random
import csv
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T
import matplotlib.pyplot as plt

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.kid import KernelInceptionDistance
except ImportError:
    print('AVISO: torchmetrics[image] não encontrado.')

# 1. Configurações de Reprodução e Caminhos
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'VAE', 'Diffusion_model', 'GAN', 'DAE', 'Exemplo'}:
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
KAGGLE_ROOT = PROJECT_ROOT / 'ArtBench-10'
TRAINING_CSV_PATH = PROJECT_ROOT / 'student_start_pack' / 'training_20_percent.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

import csv
import pickle
from datasets import Dataset as HFDataset, DatasetDict, Features, Image, ClassLabel

def _get_pickle_value(obj, key):
    if key in obj:
        return obj[key]
    bkey = key.encode("utf-8")
    if bkey in obj:
        return obj[bkey]
    raise KeyError(f"Missing key '{key}' in pickle object")



def _resolve_kaggle_paths(kaggle_root):
    root = Path(kaggle_root)
    csv_path = root / "ArtBench-10.csv"
    batch_dir = root / "artbench-10-python" / "artbench-10-batches-py"
    return root, csv_path, batch_dir



def load_kaggle_artbench10_splits(kaggle_root):
    root, csv_path, batch_dir = _resolve_kaggle_paths(kaggle_root)
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Kaggle CSV not found: {csv_path}. "
            "Expected the original ArtBench-10 folder structure."
        )
    if not batch_dir.exists():
        raise FileNotFoundError(
            f"Kaggle CIFAR batches not found: {batch_dir}. "
            "Expected ArtBench-10/artbench-10-python/artbench-10-batches-py"
        )

    with open(batch_dir / "meta", "rb") as f:
        meta = pickle.load(f)
    styles = _get_pickle_value(meta, "styles")
    if not isinstance(styles, list) or len(styles) == 0:
        raise ValueError(f"Could not read class names from {batch_dir / 'meta'}")
    styles = [str(s).strip() for s in styles]
    style_to_id = {name: i for i, name in enumerate(styles)}

    csv_label_ids = {"train": {}, "test": {}}
    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        required = {"split", "label", "cifar_index"}
        missing = required.difference(set(reader.fieldnames or []))
        if missing:
            raise ValueError(f"CSV is missing required columns {sorted(missing)}: {csv_path}")

        for row in reader:
            split = str(row.get("split", "")).strip().lower()
            if split not in csv_label_ids:
                continue

            label_name = str(row.get("label", "")).strip()
            if label_name not in style_to_id:
                raise ValueError(
                    f"Unknown label '{label_name}' in {csv_path}. "
                    f"Known labels: {styles}"
                )

            try:
                idx = int(row.get("cifar_index"))
            except Exception as exc:
                raise ValueError(f"Invalid cifar_index '{row.get('cifar_index')}' in {csv_path}") from exc

            csv_label_ids[split][idx] = int(style_to_id[label_name])

    def _load_batch(path):
        with open(path, "rb") as f:
            batch = pickle.load(f)
        data = np.asarray(_get_pickle_value(batch, "data"), dtype=np.uint8)
        labels = np.asarray(_get_pickle_value(batch, "labels"), dtype=np.int64)
        if data.ndim != 2 or data.shape[1] != 3072:
            raise ValueError(f"Unexpected data shape in {path}: {data.shape}")
        images = data.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        return images, labels

    train_images_chunks = []
    train_labels_chunks = []
    for batch_idx in range(1, 6):
        images, labels = _load_batch(batch_dir / f"data_batch_{batch_idx}")
        train_images_chunks.append(images)
        train_labels_chunks.append(labels)
    train_images = np.concatenate(train_images_chunks, axis=0)
    train_labels_raw = np.concatenate(train_labels_chunks, axis=0)
    test_images, test_labels_raw = _load_batch(batch_dir / "test_batch")

    def _labels_from_csv(split, n, labels_raw):
        ids = csv_label_ids[split]
        out = np.full((n,), -1, dtype=np.int64)
        for idx, label_id in ids.items():
            if idx < 0 or idx >= n:
                raise ValueError(
                    f"CSV {split} index {idx} out of bounds for {n} samples ({csv_path})"
                )
            out[idx] = int(label_id)
        missing = int(np.sum(out < 0))
        if missing > 0:
            raise ValueError(
                f"CSV {csv_path} is missing {missing} labels for split '{split}'."
            )
        mismatches = int(np.sum(out != labels_raw))
        if mismatches > 0:
            raise ValueError(
                f"CSV labels and batch labels disagree for {mismatches} samples in split '{split}'."
            )
        return out

    train_labels = _labels_from_csv("train", train_images.shape[0], train_labels_raw)
    test_labels = _labels_from_csv("test", test_images.shape[0], test_labels_raw)

    features = Features({
        "image": Image(),
        "label": ClassLabel(names=styles),
    })

    train_ds = HFDataset.from_dict(
        {
            "image": [train_images[i] for i in range(train_images.shape[0])],
            "label": train_labels.tolist(),
        },
        features=features,
    )
    test_ds = HFDataset.from_dict(
        {
            "image": [test_images[i] for i in range(test_images.shape[0])],
            "label": test_labels.tolist(),
        },
        features=features,
    )

    print(f"Dataset source: kaggle root='{root}'")
    return DatasetDict(train=train_ds, test=test_ds)



# 2. Definições de Dispositivo e Constantes
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
IMAGE_SIZE = 32
BATCH_SIZE = 64
NUM_WORKERS = 2

def safe_num_workers(requested: int) -> int:
    if 'ipykernel' in sys.modules and int(requested) > 0:
        return 0
    return int(requested)

# 3. Transforms (BASE e AUGMENTED)
BASE_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

AUGMENTED_TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 4. Dataset e Funções de Utilidade
class HFDatasetTorch(Dataset):
    def __init__(self, hf_split, transform=None, indices=None):
        self.ds = hf_split
        self.transform = transform
        self.indices = list(range(len(hf_split))) if indices is None else list(indices)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        ex = self.ds[real_idx]
        img = ex['image']
        y = int(ex['label'])
        x = self.transform(img) if self.transform else img
        return x, y, real_idx

def load_ids_from_training_csv(csv_path: Path, index_column: str = 'train_id_original') -> list[int]:
    if not csv_path.exists():
        raise FileNotFoundError(f'Ficheiro não encontrado: {csv_path}')
    ids = []
    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        r = csv.DictReader(f)
        for row in r:
            v = str(row.get(index_column, '')).strip()
            if v: ids.append(int(v))
    return ids

def build_loader(indices, transform, shuffle=True, batch_size=BATCH_SIZE):
    ds = HFDatasetTorch(train_hf, transform=transform, indices=indices)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )

def plot_loss_curves(history, title):
    plt.figure(figsize=(8, 4))
    for key, values in history.items():
        if values and isinstance(values[0], (int, float)):
            plt.plot(values, label=key)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# 5. Carregar Dados Reais e Criar Loaders
print(f'Lendo ArtBench-10 de: {KAGGLE_ROOT}...')
try:
    hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
    train_hf = hf_ds['train']
    test_hf = hf_ds['test']
    class_names = list(train_hf.features['label'].names)

    subset_ids = load_ids_from_training_csv(TRAINING_CSV_PATH)
    full_train_ids = list(range(len(train_hf)))

    dev_loader = build_loader(subset_ids, BASE_TRANSFORM, shuffle=True)
    dev_loader_aug = build_loader(subset_ids, AUGMENTED_TRANSFORM, shuffle=True)
    full_train_loader = build_loader(full_train_ids, BASE_TRANSFORM, shuffle=True)
    full_train_loader_aug = build_loader(full_train_ids, AUGMENTED_TRANSFORM, shuffle=True)
    test_loader = DataLoader(
        HFDatasetTorch(test_hf, transform=BASE_TRANSFORM),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=safe_num_workers(NUM_WORKERS),
        pin_memory=torch.cuda.is_available(),
    )
    print('--- Setup Concluído com Sucesso ---')
except Exception as e:
    print(f'Erro ao carregar dados: {e}')
print('Dispositivo:', device)


## 2. Imports do pipeline de maturidade

A celula seguinte importa o modulo `cgan_maturity_study.py`, que concentra a logica de treino, grid search, ablacao e avaliacao. Esta separacao traz duas vantagens:

- reduz a duplicacao de codigo entre experiencias;
- permite reutilizar exatamente a mesma implementacao em diferentes fases, melhorando a rastreabilidade dos resultados.

Na pratica, o notebook funciona como uma narrativa experimental e o modulo externo funciona como backend do pipeline. Isto ajuda bastante quando queremos perceber se uma diferenca de metrica vem de uma alteracao arquitetural real ou apenas de uma variacao acidental no codigo.


In [ ]:
# ============================================================
# CGAN Pipeline (full code, inlined from cgan_maturity_study.py)
# ============================================================
import gc
import itertools
import json
from dataclasses import dataclass
from typing import Dict, List, Tuple
from types import SimpleNamespace
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.utils import save_image, make_grid

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class HFDatasetTorch(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.images = []
        self.labels = []
        self.transform = transform
        print("Loading dataset into RAM...")
        for item in tqdm(hf_dataset, desc="Loading"):
            image = item["image"].convert("RGB")
            label = int(item["label"])
            if self.transform is not None:
                image = self.transform(image)
            self.images.append(image)
            self.labels.append(label)
        print(f"Loaded {len(self.images)} images.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx], idx


class CGANGenerator(nn.Module):
    def __init__(self, latent_dim=100, embed_dim=50, num_classes=10, base_ch=64):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        in_dim = latent_dim + embed_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(in_dim, base_ch * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(base_ch * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_ch * 4, base_ch * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_ch, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        emb = self.label_emb(labels).unsqueeze(-1).unsqueeze(-1)
        x = torch.cat([z, emb], dim=1)
        return self.net(x)


class CGANDiscriminator(nn.Module):
    def __init__(self, num_classes=10, embed_dim=50, base_ch=64, img_size=32):
        super().__init__()
        sn = nn.utils.spectral_norm
        self.label_emb = nn.Embedding(num_classes, embed_dim)
        self.label_proj = nn.Linear(embed_dim, img_size * img_size)
        self.img_size = img_size
        self.net = nn.Sequential(
            sn(nn.Conv2d(4, base_ch, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(base_ch, base_ch * 2, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(base_ch * 2, base_ch * 4, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            sn(nn.Conv2d(base_ch * 4, 1, 4, 1, 0, bias=False)),
        )

    def forward(self, img, labels):
        emb = self.label_emb(labels)
        label_map = self.label_proj(emb).view(-1, 1, self.img_size, self.img_size)
        x = torch.cat([img, label_map], dim=1)
        return self.net(x).view(-1, 1)


def init_weights(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1 and hasattr(m, "weight"):
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


def denormalize(t):
    return (t * 0.5 + 0.5).clamp(0, 1)


@dataclass
class TrainConfig:
    base_channels: int
    latent_dim: int
    embed_dim: int
    epochs: int
    lr_g: float
    lr_d: float
    n_critic: int
    label_smooth: float


def sample_cgan(generator, n_samples, latent_dim, num_classes, device, seed=None, rng=None):
    if rng is None and seed is not None:
        rng = torch.Generator(device=device)
        rng.manual_seed(int(seed))
    if rng is None:
        z = torch.randn(n_samples, latent_dim, 1, 1, device=device)
        labels = torch.randint(0, num_classes, (n_samples,), device=device)
    else:
        z = torch.randn(n_samples, latent_dim, 1, 1, generator=rng, device=device)
        labels = torch.randint(0, num_classes, (n_samples,), generator=rng, device=device)
    generator.eval()
    with torch.no_grad():
        out = generator(z, labels)
    return denormalize(out)


@torch.no_grad()
def quick_fid(generator, real_loader, latent_dim, num_classes, device, n_samples=2000, seed=999):
    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    rng = torch.Generator(device=device)
    rng.manual_seed(int(seed))
    seen = 0
    for real, _, _ in real_loader:
        real = real.to(device)
        bsz = min(real.size(0), n_samples - seen)
        fid.update(denormalize(real[:bsz]), real=True)
        fake = sample_cgan(generator, bsz, latent_dim, num_classes, device, rng=rng)
        fid.update(fake, real=False)
        seen += bsz
        if seen >= n_samples:
            break
    return float(fid.compute().detach().cpu())


def train_cgan(
    train_loader_aug,
    eval_loader,
    config: TrainConfig,
    run_dir: Path,
    device,
    num_classes=10,
    fid_every=10,
    save_every=5,
):
    run_dir.mkdir(parents=True, exist_ok=True)
    g_model = CGANGenerator(
        latent_dim=config.latent_dim,
        embed_dim=config.embed_dim,
        num_classes=num_classes,
        base_ch=config.base_channels,
    ).to(device)
    d_model = CGANDiscriminator(
        num_classes=num_classes,
        embed_dim=config.embed_dim,
        base_ch=config.base_channels,
    ).to(device)
    g_model.apply(init_weights)
    d_model.apply(init_weights)

    criterion = nn.BCEWithLogitsLoss()
    opt_g = torch.optim.AdamW(
        g_model.parameters(),
        lr=config.lr_g,
        betas=(0.5, 0.999),
        weight_decay=1e-4,
    )
    opt_d = torch.optim.AdamW(
        d_model.parameters(),
        lr=config.lr_d,
        betas=(0.5, 0.999),
        weight_decay=1e-4,
    )

    history = {"d_loss": [], "g_loss": [], "fid_quick": []}
    best_fid = float("inf")

    for epoch in range(1, config.epochs + 1):
        g_model.train()
        d_model.train()
        total_d = 0.0
        total_g = 0.0
        progress = tqdm(train_loader_aug, desc=f"{run_dir.name} {epoch}/{config.epochs}", leave=False)

        for real, labels, _ in progress:
            real = real.to(device)
            labels = labels.to(device)
            bsz = real.size(0)

            real_lbl = config.label_smooth * torch.ones(bsz, 1, device=device)
            fake_lbl = torch.zeros(bsz, 1, device=device)

            opt_d.zero_grad(set_to_none=True)
            loss_real = criterion(d_model(real, labels), real_lbl)
            z = torch.randn(bsz, config.latent_dim, 1, 1, device=device)
            fake = g_model(z, labels)
            loss_fake = criterion(d_model(fake.detach(), labels), fake_lbl)
            loss_d = 0.5 * (loss_real + loss_fake)
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(d_model.parameters(), 1.0)
            opt_d.step()

            g_sum = 0.0
            for _ in range(config.n_critic):
                opt_g.zero_grad(set_to_none=True)
                z2 = torch.randn(bsz, config.latent_dim, 1, 1, device=device)
                fake2 = g_model(z2, labels)
                loss_g = criterion(d_model(fake2, labels), torch.ones(bsz, 1, device=device))
                loss_g.backward()
                torch.nn.utils.clip_grad_norm_(g_model.parameters(), 1.0)
                opt_g.step()
                g_sum += float(loss_g.item())

            total_d += float(loss_d.item())
            total_g += g_sum / config.n_critic
            progress.set_postfix(d=f"{loss_d.item():.4f}", g=f"{(g_sum / config.n_critic):.4f}")

        avg_d = total_d / max(1, len(train_loader_aug))
        avg_g = total_g / max(1, len(train_loader_aug))
        history["d_loss"].append(avg_d)
        history["g_loss"].append(avg_g)

        line = f"Epoch {epoch:03d}/{config.epochs} | d_loss={avg_d:.4f} | g_loss={avg_g:.4f}"
        if epoch % fid_every == 0:
            fid_val = quick_fid(
                g_model,
                eval_loader,
                latent_dim=config.latent_dim,
                num_classes=num_classes,
                device=device,
                n_samples=2000,
                seed=999,
            )
            history["fid_quick"].append({"epoch": epoch, "fid": fid_val})
            line += f" | FID_quick={fid_val:.2f}"
            if fid_val < best_fid:
                best_fid = fid_val
                torch.save(g_model.state_dict(), run_dir / "cgan_generator_best.pt")
                torch.save(d_model.state_dict(), run_dir / "cgan_discriminator_best.pt")
                line += " <- BEST"
            g_model.train()
        print(line)

        if epoch % save_every == 0 or epoch == config.epochs:
            g_model.eval()
            with torch.no_grad():
                imgs_per_class = []
                for cls in range(num_classes):
                    labels = torch.full((3,), cls, device=device, dtype=torch.long)
                    z = torch.randn(3, config.latent_dim, 1, 1, device=device)
                    imgs = g_model(z, labels)
                    imgs_per_class.append(denormalize(imgs).cpu())
                grid = torch.cat(imgs_per_class, dim=0)
                save_image(
                    make_grid(grid, nrow=num_classes),
                    run_dir / f"cgan_samples_epoch_{epoch:03d}.png",
                )

    torch.save(g_model.state_dict(), run_dir / "cgan_generator_last.pt")
    torch.save(d_model.state_dict(), run_dir / "cgan_discriminator_last.pt")
    if not (run_dir / "cgan_generator_best.pt").exists():
        torch.save(g_model.state_dict(), run_dir / "cgan_generator_best.pt")

    with open(run_dir / "cgan_history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    return history, best_fid


def stability_score(history: Dict[str, List[float]]) -> float:
    if not history["d_loss"] or not history["g_loss"]:
        return float("inf")
    nash = 0.69314718056
    return abs(history["d_loss"][-1] - nash) + abs(history["g_loss"][-1] - nash)


def plot_ablation(histories: Dict[str, Dict[str, List[float]]], out_path: Path):
    plt.figure(figsize=(10, 6))
    for key, hist in histories.items():
        plt.plot(hist["d_loss"], label=key, linewidth=2 if "V4" in key else 1.6)
    plt.axhline(0.693, color="black", linestyle="--", label="Nash (0.693)")
    plt.xlabel("Epoch")
    plt.ylabel("Discriminator Loss")
    plt.title("CGAN Ablation: Convergence of D Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
    plt.close()


def evaluate_final(
    run_dir: Path,
    eval_loader: DataLoader,
    config: TrainConfig,
    device,
    num_classes=10,
    repeats=10,
    n_gen=5000,
):
    model = CGANGenerator(
        latent_dim=config.latent_dim,
        embed_dim=config.embed_dim,
        num_classes=num_classes,
        base_ch=config.base_channels,
    ).to(device)
    model.load_state_dict(torch.load(run_dir / "cgan_generator_best.pt", map_location=device))
    model.eval()

    results = {"fid": [], "kid": []}
    print(f"\n=== FINAL EVAL ({repeats} seeds, {n_gen} samples) ===")

    for seed in range(repeats):
        print(f"Seed {seed}/{repeats - 1}...", end=" ")
        torch.manual_seed(seed)
        np.random.seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        fid_m = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
        kid_m = KernelInceptionDistance(
            feature=2048,
            normalize=True,
            subset_size=100,
            subsets=50,
        ).to(device)

        generated = 0
        while generated < n_gen:
            bsz = min(256, n_gen - generated)
            z = torch.randn(bsz, config.latent_dim, 1, 1, device=device)
            labels = torch.randint(0, num_classes, (bsz,), device=device)
            with torch.no_grad():
                fake = denormalize(model(z, labels))
            fid_m.update(fake, real=False)
            kid_m.update(fake, real=False)
            generated += bsz

        seen_real = 0
        for real, _, _ in eval_loader:
            real = denormalize(real.to(device))
            bsz = min(real.size(0), n_gen - seen_real)
            fid_m.update(real[:bsz], real=True)
            kid_m.update(real[:bsz], real=True)
            seen_real += bsz
            if seen_real >= n_gen:
                break

        fid_val = float(fid_m.compute().detach().cpu())
        kid_val = float(kid_m.compute()[0].detach().cpu())
        print(f"FID={fid_val:.2f}, KID={kid_val:.4f}")
        results["fid"].append(fid_val)
        results["kid"].append(kid_val)

    with open(run_dir / "cgan_eval_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    fid_arr = np.array(results["fid"])
    kid_arr = np.array(results["kid"])
    print("\n=== FINAL RESULT ===")
    print(f"FID: {fid_arr.mean():.2f} +/- {fid_arr.std():.2f}")
    print(f"KID: {kid_arr.mean():.4f} +/- {kid_arr.std():.4f}")
    return results


def build_loaders(device_batch_size: int, seed: int, use_full_data: bool):

    hf_ds = load_kaggle_artbench10_splits(KAGGLE_ROOT)
    train_data = hf_ds["train"]

    if not use_full_data:
        train_data = train_data.shuffle(seed=seed).select(range(int(0.2 * len(train_data))))

    train_data = train_data.shuffle(seed=seed)

    transform_aug = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    )
    transform_norm = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ]
    )

    train_aug = HFDatasetTorch(train_data, transform_aug)
    train_norm = HFDatasetTorch(train_data, transform_norm)
    loader_aug = DataLoader(
        train_aug, batch_size=device_batch_size, shuffle=True, drop_last=True, num_workers=0
    )
    loader_norm = DataLoader(
        train_norm, batch_size=device_batch_size, shuffle=False, num_workers=0
    )
    return loader_aug, loader_norm


def run_grid_search(args, device):
    out_dir = OUTPUT_ROOT / "cgan_gridsearch_maturity"
    out_dir.mkdir(parents=True, exist_ok=True)
    loader_aug, loader_norm = build_loaders(args.batch_size, args.seed, use_full_data=False)

    grid_base = parse_csv_numbers(args.grid_base_channels, int)
    grid_lrg = parse_csv_numbers(args.grid_lr_g, float)
    grid_latent = parse_csv_numbers(args.grid_latent_dims, int)
    grid_embed = parse_csv_numbers(args.grid_embed_dims, int)
    results = []

    all_combos = list(itertools.product(grid_base, grid_lrg, grid_latent, grid_embed))
    if args.max_grid_runs > 0:
        all_combos = all_combos[: args.max_grid_runs]
    print(f"\n=== CGAN GRID SEARCH ({len(all_combos)} runs) ===")
    print(
        f"base={grid_base} | lr_g={grid_lrg} | latent={grid_latent} | embed={grid_embed}"
    )
    for base_ch, lr_g, latent_dim, embed_dim in all_combos:
        run_name = (
            f"grid_base-{base_ch}_lrg-{lr_g}_latent-{latent_dim}_embed-{embed_dim}"
            .replace(".", "p")
        )
        run_dir = out_dir / run_name
        cfg = TrainConfig(
            base_channels=base_ch,
            latent_dim=latent_dim,
            embed_dim=embed_dim,
            epochs=args.grid_epochs,
            lr_g=lr_g,
            lr_d=args.lr_d,
            n_critic=args.n_critic,
            label_smooth=args.label_smooth,
        )
        print(f"\nRunning {run_name}")
        hist, best_fid = train_cgan(
            train_loader_aug=loader_aug,
            eval_loader=loader_norm,
            config=cfg,
            run_dir=run_dir,
            device=device,
            fid_every=args.fid_every,
            save_every=args.save_every,
        )
        score = stability_score(hist)
        fid_last = hist["fid_quick"][-1]["fid"] if hist["fid_quick"] else None
        row = {
            "params": {
                "base_channels": base_ch,
                "lr_g": lr_g,
                "lr_d": args.lr_d,
                "latent_dim": latent_dim,
                "embed_dim": embed_dim,
            },
            "score": score,
            "best_fid_quick": best_fid,
            "last_fid_quick": fid_last,
            "run_dir": str(run_dir),
        }
        results.append(row)
        print(f"Score={score:.4f} | best FID_quick={best_fid:.2f}")

    results.sort(key=lambda x: x["score"])
    with open(out_dir / "grid_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print("\nTop 3 by stability score:")
    for row in results[:3]:
        print(row)
    return results


def run_ablation(args, device):
    out_dir = OUTPUT_ROOT / "cgan_ablation_maturity"
    out_dir.mkdir(parents=True, exist_ok=True)
    loader_aug, loader_norm = build_loaders(args.batch_size, args.seed, use_full_data=False)

    variants = [
        (
            "V1_Vanilla",
            TrainConfig(64, 128, 50, args.ablation_epochs, 2e-4, 2e-4, 1, 1.0),
        ),
        (
            "V2_TTUR",
            TrainConfig(64, 128, 50, args.ablation_epochs, 2e-4, 1e-4, 1, 1.0),
        ),
        (
            "V3_TTUR_LS",
            TrainConfig(64, 128, 50, args.ablation_epochs, 2e-4, 1e-4, 1, 0.9),
        ),
        (
            "V4_Full",
            TrainConfig(64, 128, 50, args.ablation_epochs, 2e-4, 1e-4, 2, 0.9),
        ),
    ]

    histories = {}
    summary = []
    print("\n=== CGAN ABLATION STUDY ===")
    for name, cfg in variants:
        run_dir = out_dir / name
        print(f"\n{name}")
        hist, best_fid = train_cgan(
            train_loader_aug=loader_aug,
            eval_loader=loader_norm,
            config=cfg,
            run_dir=run_dir,
            device=device,
            fid_every=args.fid_every,
            save_every=args.save_every,
        )
        histories[name] = hist
        summary.append(
            {
                "variant": name,
                "stability_score": stability_score(hist),
                "best_fid_quick": best_fid,
                "final_d_loss": hist["d_loss"][-1],
                "final_g_loss": hist["g_loss"][-1],
                "run_dir": str(run_dir),
            }
        )

    plot_ablation(histories, out_dir / "cgan_ablation_d_loss.png")
    summary.sort(key=lambda x: x["stability_score"])
    with open(out_dir / "ablation_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("\nAblation ranking:")
    for row in summary:
        print(row)
    return summary



def parse_csv_numbers(raw: str, cast_fn):
    values = []
    for token in str(raw).split(","):
        token = token.strip()
        if not token:
            continue
        values.append(cast_fn(token))
    if not values:
        raise ValueError(f"Invalid CSV list: '{raw}'")
    return values



DEVICE = device  # alias used by grid/ablation cells
print('Pipeline CGAN carregado com sucesso.')
print('Device:', DEVICE)


## 3. Arquitetura e Configuracao do CGAN

O CGAN estende o esquema DCGAN ao introduzir informacao de classe tanto no generator como no discriminator.

**Generator condicional**
- parte de um vetor latente `z` concatenado com um `embedding` da label;
- usa blocos `ConvTranspose + BatchNorm + ReLU` ate produzir imagens RGB `32 x 32`;
- tenta modelar nao apenas a manifold global das obras, mas tambem a coerencia com o estilo pedido.

**Discriminator condicional**
- recebe a imagem juntamente com um mapa espacial derivado da label;
- usa `Spectral Normalization` e `LeakyReLU`, tal como no DCGAN estabilizado;
- aprende a distinguir simultaneamente real/fake e compatibilidade imagem-label.

Isto torna o problema mais exigente: o modelo passa a resolver uma tarefa mais estruturada, mas tambem fragmenta a densidade estatistica por classe. Em datasets artistico-estilisticos, esse trade-off pode ajudar no controlo semantico mas piorar a qualidade global se a capacidade do modelo nao for suficiente.


### 3.1 Melhorias manuais e adaptacao condicional do CGAN

Ao contrario do DCGAN, o CGAN nao tem uma celula separada de "melhorias manuais". Isto foi uma decisao metodologica: o modelo condicional foi construido diretamente sobre a receita estabilizada que ja tinha sido validada no DCGAN.

Na pratica, as melhorias que no DCGAN aparecem como uma fase incremental foram incorporadas logo no ponto de partida do CGAN:

| Componente herdado/adaptado | Como aparece no CGAN | Objetivo |
| :--- | :--- | :--- |
| `Tanh` + normalizacao `[-1, 1]` | saida do `CGANGenerator` | alinhar geracao e dados reais |
| `SpectralNorm` | convolucoes do `CGANDiscriminator` | limitar explosao do discriminador |
| `TTUR` | `lr_g > lr_d` nas configuracoes finais | evitar dominancia do discriminador |
| `Label smoothing` | `label_smooth=0.9` | reduzir overconfidence nas labels reais |
| `n_critic=2` | duas atualizacoes do generator por ciclo | reforcar resposta do generator |
| `AdamW` + gradient clipping | pipeline de treino | estabilizar otimizacao adversarial |

A novidade especifica do CGAN e o condicionamento por classe: o generator recebe um embedding da label concatenado ao vetor latente, enquanto o discriminator recebe a imagem acompanhada de um mapa espacial derivado da label. Por isso, a validacao das melhorias aparece sobretudo na grid search e na ablacao: essas secoes testam se a receita estabilizada do DCGAN continua a fazer sentido quando o problema passa de geracao incondicional para geracao condicionada por estilo.

In [ ]:
# Main experiment config used by both grid and ablation
# Tip: for a fast smoke test, set grid_epochs/ablation_epochs to 1 and max_grid_runs to 1.
args = SimpleNamespace(
    seed=42,
    batch_size=128,
    grid_epochs=15,
    ablation_epochs=15,
    fid_every=10,
    save_every=5,
    label_smooth=0.9,
    n_critic=2,
    lr_d=1e-4,
    grid_base_channels="64,96",
    grid_lr_g="2e-4,1.5e-4",
    grid_latent_dims="100,128",
    grid_embed_dims="50,100",
    max_grid_runs=0,
)

set_seed(args.seed)
args


## 4. Otimizacao de Hiperparametros: Grid Search (Subset 20%)

A grid search do CGAN foi desenhada para responder a quatro perguntas:
- quanta largura (`base_channels`) a variante condicional precisa para nao colapsar cedo;
- qual a `learning rate` do generator mais favoravel em regime TTUR;
- se um `latent_dim` mais pequeno facilita ou limita a diversidade;
- qual o tamanho de `embedding` suficiente para transmitir a label sem inflacionar desnecessariamente o modelo.

Como no DCGAN, o subset de 20% serve apenas para desenvolvimento. O objetivo aqui e ordenar configuracoes de forma barata e encontrar uma regiao credivel do espaco de procura antes do treino longo.

O `quick FID` desta fase deve ser lido como **heuristica de progresso**. Ele ajuda a distinguir configuracoes promissoras das claramente instaveis, mas nao deve ser usado isoladamente como resultado final de comparacao com outras familias.


In [ ]:
grid_results_file = OUTPUT_ROOT / 'cgan_gridsearch_maturity' / 'grid_results.json'
if grid_results_file.exists():
    grid_results = json.loads(grid_results_file.read_text('utf-8'))
    print(f'Grid Search carregado de: {grid_results_file}')
else:
    grid_results = run_grid_search(args, DEVICE)

grid_results[:3]


# --- Visualizar resultados da Grid Search ---
if grid_results:
    print('\n=== RESULTADOS GRID SEARCH ===')
    for r in sorted(grid_results, key=lambda x: x['best_fid_quick']):
        print(f"  base={r['params']['base_channels']}, lr_g={r['params']['lr_g']}, "
              f"latent={r['params']['latent_dim']}, embed={r['params']['embed_dim']} "
              f"-> FID={r['best_fid_quick']:.2f}, Score={r.get('score', r.get('stability_score', 0)):.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    names = [f"b{r['params']['base_channels']}\nlr{r['params']['lr_g']}" for r in grid_results]
    fids = [r['best_fid_quick'] for r in grid_results]
    scores = [r.get('score', r.get('stability_score', 0)) for r in grid_results]

    axes[0].plot(range(len(fids)), fids, 'o-', color='#9b59b6', linewidth=2, markersize=6)
    axes[0].set_xticks(range(len(names)))
    axes[0].set_xticklabels(names, fontsize=7, rotation=30, ha='right')
    axes[0].set_ylabel('Quick FID (lower is better)')
    axes[0].set_title('Grid Search: FID por Configuracao')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(range(len(scores)), scores, 'o-', color='#e67e22', linewidth=2, markersize=6)
    axes[1].set_xticks(range(len(names)))
    axes[1].set_xticklabels(names, fontsize=7, rotation=30, ha='right')
    axes[1].set_ylabel('Stability Score (lower is better)')
    axes[1].set_title('Grid Search: Estabilidade')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('CGAN Grid Search Results (20% subset, 15 epochs)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Amostras visuais: melhor e pior config do Grid Search ---
import json as _json
from pathlib import Path as _Path


def _load_torch_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _first_existing_checkpoint(run_dir, names):
    for name in names:
        candidate = run_dir / name
        if candidate.exists():
            return candidate
    return run_dir / names[0]


def _make_rng(seed):
    try:
        rng = torch.Generator(device=device)
    except (TypeError, RuntimeError):
        rng = torch.Generator()
    rng.manual_seed(int(seed))
    return rng


grid_file = OUTPUT_ROOT / 'cgan_gridsearch_maturity' / 'grid_results.json'
configs_to_show = []
if grid_file.exists():
    _gr = _json.loads(grid_file.read_text('utf-8'))
    if _gr:
        _sorted = sorted(_gr, key=lambda x: x['best_fid_quick'])
        configs_to_show = [('Melhor Config', _sorted[0]), ('Pior Config', _sorted[-1])]
    else:
        print('grid_results.json esta vazio, a saltar amostras do grid search.')
else:
    print('grid_results.json nao encontrado, a saltar amostras do grid search.')

if configs_to_show:
    class_names = ['art_nouveau', 'baroque', 'expressionism', 'impressionism', 'post_impressionism',
                   'realism', 'renaissance', 'romanticism', 'surrealism', 'ukiyo_e']

    fig, all_axes = plt.subplots(len(configs_to_show), len(class_names), figsize=(20, 5), squeeze=False)

    for row, (label, entry) in enumerate(configs_to_show):
        p = entry['params']
        run_dir = _Path(entry['run_dir'])
        ckpt = _first_existing_checkpoint(
            run_dir,
            ['cgan_generator_best.pt', 'cgan_generator_last.pt', 'generator_best.pt'],
        )
        if not ckpt.exists():
            print(f'{label}: checkpoint nao encontrado em {ckpt}')
            for cls in range(10):
                all_axes[row, cls].axis('off')
            continue

        G = CGANGenerator(
            latent_dim=p['latent_dim'], embed_dim=p.get('embed_dim', 50),
            num_classes=10, base_ch=p['base_channels'],
        ).to(device)
        G.load_state_dict(_load_torch_state_dict(ckpt, device))
        G.eval()

        with torch.no_grad():
            for cls in range(10):
                rng = _make_rng(20260425 + cls)
                z = torch.randn(1, p['latent_dim'], 1, 1, generator=rng, device=device)
                lbl = torch.tensor([cls], device=device)
                img = denormalize(G(z, lbl)).cpu()[0]
                all_axes[row, cls].imshow(img.permute(1, 2, 0).numpy())
                all_axes[row, cls].axis('off')
                if row == 0:
                    all_axes[row, cls].set_title(class_names[cls], fontsize=7)
        cfg_str = f"b={p['base_channels']}, z={p['latent_dim']}, e={p.get('embed_dim', 50)}"
        all_axes[row, 0].set_ylabel(f'{label}\n{cfg_str}\nFID={entry["best_fid_quick"]:.0f}',
                                     fontsize=7, rotation=0, labelpad=80)
        del G
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    plt.suptitle('CGAN Grid Search - Amostras (1 por classe, melhor vs pior config)', fontsize=13, y=1.05)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_ROOT / 'cgan_gridsearch_maturity' / 'grid_search_samples.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Amostras do Grid Search geradas com sucesso.')

## 5. Estudo de Ablacao de Estabilidade (Subset 20%)

A ablacao verifica se as mesmas tecnicas que estabilizaram o DCGAN tambem fazem sentido no caso condicional. As variantes mantem uma progressao simples:
- `V1`: treino vanilla;
- `V2`: apenas `TTUR`;
- `V3`: `TTUR + label smoothing`;
- `V4`: configuracao completa com `n_critic = 2`.

O interesse desta secao vai alem do ranking final. Num CGAN, a dinamica pode parecer razoavel em termos de loss media e, ainda assim, gerar artefactos repetitivos por classe ou checkerboard patterns. A ablacao ajuda-nos a perceber se a estabilidade observada no treino longo e uma consequencia robusta do metodo ou apenas um acaso de inicializacao.


In [ ]:
ablation_file = OUTPUT_ROOT / 'cgan_ablation_maturity' / 'ablation_summary.json'
if ablation_file.exists():
    ablation_summary = json.loads(ablation_file.read_text('utf-8'))
    print(f'Ablacao carregada de: {ablation_file}')
else:
    ablation_summary = run_ablation(args, DEVICE)

ablation_summary


# --- Visualizar resultados da Ablacao ---
if ablation_summary:
    print('\n=== RESULTADOS ABLACAO ===')
    variant_names = []
    variant_fids = []
    variant_scores = []
    for r in ablation_summary:
        name = r['variant']
        fid = r['best_fid_quick']
        score = r['stability_score']
        variant_names.append(name)
        variant_fids.append(fid)
        variant_scores.append(score)
        print(f"  {name}: FID={fid:.2f}, Score={score:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(variant_names, variant_fids, 'o-', color='#3498db', linewidth=2, markersize=8)
    axes[0].set_ylabel('Quick FID')
    axes[0].set_title('FID por Variante')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(variant_names, variant_scores, 'o-', color='#e67e22', linewidth=2, markersize=8)
    axes[1].set_ylabel('Stability Score')
    axes[1].set_title('Estabilidade por Variante')
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('CGAN Ablation Study (20% subset, 15 epochs)', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Amostras visuais: variantes da Ablacao ---
import json as _json
from pathlib import Path as _Path


def _load_torch_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _first_existing_checkpoint(run_dir, names):
    for name in names:
        candidate = run_dir / name
        if candidate.exists():
            return candidate
    return run_dir / names[0]


def _make_rng(seed):
    try:
        rng = torch.Generator(device=device)
    except (TypeError, RuntimeError):
        rng = torch.Generator()
    rng.manual_seed(int(seed))
    return rng


class_names = ['art_nouveau', 'baroque', 'expressionism', 'impressionism', 'post_impressionism',
               'realism', 'renaissance', 'romanticism', 'surrealism', 'ukiyo_e']

ablation_root = OUTPUT_ROOT / 'cgan_ablation_maturity'
summary_file = ablation_root / 'ablation_summary.json'
summary_by_variant = {}
if summary_file.exists():
    for row in _json.loads(summary_file.read_text('utf-8')):
        summary_by_variant[row['variant']] = row

ablation_variants = [
    ('V1_Vanilla', dict(base_channels=64, latent_dim=128, embed_dim=50)),
    ('V2_TTUR', dict(base_channels=64, latent_dim=128, embed_dim=50)),
    ('V3_TTUR_LS', dict(base_channels=64, latent_dim=128, embed_dim=50)),
    ('V4_Full', dict(base_channels=64, latent_dim=128, embed_dim=50)),
]

fig, all_axes = plt.subplots(len(ablation_variants), len(class_names), figsize=(20, 9), squeeze=False)

for row, (variant, params) in enumerate(ablation_variants):
    info = summary_by_variant.get(variant, {})
    run_dir = _Path(info.get('run_dir', str(ablation_root / variant)))
    ckpt = _first_existing_checkpoint(
        run_dir,
        ['cgan_generator_best.pt', 'cgan_generator_last.pt', 'generator_best.pt'],
    )
    if not ckpt.exists():
        print(f'{variant}: checkpoint nao encontrado em {ckpt}')
        for cls in range(10):
            all_axes[row, cls].axis('off')
        all_axes[row, 0].set_ylabel(variant, fontsize=9, rotation=0, labelpad=65)
        continue

    G = CGANGenerator(
        latent_dim=params['latent_dim'], embed_dim=params['embed_dim'],
        num_classes=10, base_ch=params['base_channels'],
    ).to(device)
    G.load_state_dict(_load_torch_state_dict(ckpt, device))
    G.eval()

    with torch.no_grad():
        for cls in range(10):
            rng = _make_rng(20260525 + cls)
            z = torch.randn(1, params['latent_dim'], 1, 1, generator=rng, device=device)
            lbl = torch.tensor([cls], device=device)
            img = denormalize(G(z, lbl)).cpu()[0]
            all_axes[row, cls].imshow(img.permute(1, 2, 0).numpy())
            all_axes[row, cls].axis('off')
            if row == 0:
                all_axes[row, cls].set_title(class_names[cls], fontsize=7)

    fid_text = ''
    if 'best_fid_quick' in info:
        fid_text = f'\nFID={info["best_fid_quick"]:.0f}'
    all_axes[row, 0].set_ylabel(f'{variant}{fid_text}', fontsize=8, rotation=0, labelpad=70)
    del G
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

plt.suptitle('CGAN Ablacao - Amostras por Variante (1 por classe)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(str(ablation_root / 'ablation_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Amostras da Ablacao geradas com sucesso.')

## 6. Treino final no conjunto completo (400 epocas)

Configuracao final: `base_channels=128`, `latent_dim=100`, `embed_dim=50`, `lr_g=1.5e-4`, `lr_d=1e-4`, `n_critic=2`, `label_smooth=0.9`.

Treinamos 400 epocas (em vez de 200) porque o FID estava ainda a melhorar no final das 200 epocas.
Usa `build_loaders()` para pre-carregar o dataset completo em RAM e `train_cgan()` para o treino.

**A celula seguinte produz diretamente**: graficos de loss/FID, amostras geradas por classe.


In [ ]:
import gc
# ============================================================
# Importar TODAS as dependencias para a celula de treino final
# ============================================================
import json
from tqdm.auto import tqdm
from torchvision.utils import save_image, make_grid
# (Models and utilities already defined above)
# Funcao de avaliacao rapida de FID usada durante o treino
@torch.no_grad()
def cgan_quick_fid(generator, real_loader, latent_dim=100, n_samples=2000, seed=999):
    fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    seen = 0
    for real, _, _ in real_loader:
        real = real.to(device)
        batch = min(real.size(0), n_samples - seen)
        fid_metric.update(denormalize(real[:batch]), real=True)
        z = torch.randn(batch, latent_dim, 1, 1, device=device)
        labels = torch.randint(0, 10, (batch,), device=device)
        generator.eval()
        fake = denormalize(generator(z, labels))
        fid_metric.update(fake, real=False)
        seen += batch
        if seen >= n_samples:
            break
    return float(fid_metric.compute().detach().cpu())

print('Todas as dependencias importadas com sucesso.')


In [ ]:
# ============================================================
# TREINO FINAL CGAN (400 epocas, dataset completo)
# Reutiliza checkpoints existentes quando ja foram gerados.
# ============================================================
import gc
import json


def _load_torch_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _make_rng(seed):
    try:
        rng = torch.Generator(device=device)
    except (TypeError, RuntimeError):
        rng = torch.Generator()
    rng.manual_seed(int(seed))
    return rng


set_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

config = TrainConfig(
    base_channels=128, latent_dim=100, embed_dim=50,
    epochs=400, lr_g=1.5e-4, lr_d=1e-4, n_critic=2, label_smooth=0.9,
)

run_dir = OUTPUT_ROOT / 'cgan_final_full_artbench10'
run_dir.mkdir(parents=True, exist_ok=True)
history_file = run_dir / 'cgan_history.json'
best_ckpt = run_dir / 'cgan_generator_best.pt'

print('=== TREINO FINAL CGAN (400 epocas, dataset completo) ===')
print('Config: base_ch=128, latent=100, embed=50, lr_g=1.5e-4, lr_d=1e-4')
print(f'Output: {run_dir}')

if history_file.exists() and best_ckpt.exists():
    print('Checkpoint final encontrado; a carregar historico guardado sem re-treino.')
    history = json.loads(history_file.read_text('utf-8'))
    fid_entries = history.get('fid_quick', [])
    fid_values = [float(x['fid'] if isinstance(x, dict) else x) for x in fid_entries]
    best_fid = min(fid_values) if fid_values else float('inf')
else:
    train_loader, eval_loader = build_loaders(
        device_batch_size=128, seed=42, use_full_data=True,
    )
    history, best_fid = train_cgan(
        train_loader, eval_loader, config, run_dir, device,
        fid_every=10, save_every=5,
    )

# --- GRAFICOS DE TREINO (inline, como Autoencoders) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['d_loss'], label='D Loss', color='#e74c3c', alpha=0.8)
axes[0].plot(history['g_loss'], label='G Loss', color='#3498db', alpha=0.8)
axes[0].axhline(0.693, color='black', linestyle='--', alpha=0.5, label='Nash (0.693)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('CGAN Training Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

fid_data = history.get('fid_quick', [])
if fid_data:
    if isinstance(fid_data[0], dict):
        fid_epochs = [x['epoch'] for x in fid_data]
        fid_vals = [x['fid'] for x in fid_data]
    else:
        fid_epochs = list(range(10, 10 * len(fid_data) + 1, 10))
        fid_vals = fid_data
    best_idx = fid_vals.index(min(fid_vals))
    axes[1].plot(fid_epochs, fid_vals, 'o-', color='#9b59b6', markersize=3)
    axes[1].scatter([fid_epochs[best_idx]], [fid_vals[best_idx]], color='red', s=100, zorder=5,
                    label=f'Best: {min(fid_vals):.2f} (ep {fid_epochs[best_idx]})')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID (2000 samples)')
    axes[1].set_title('Quick-FID Evolution')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Sem dados de FID', ha='center', va='center')
    axes[1].set_axis_off()

axes[2].hist(history['d_loss'][-50:], bins=20, color='#e74c3c', alpha=0.7, label='D Loss (last 50)')
axes[2].hist(history['g_loss'][-50:], bins=20, color='#3498db', alpha=0.7, label='G Loss (last 50)')
axes[2].axvline(0.693, color='black', linestyle='--', label='Nash')
axes[2].set_title('Loss Distribution (last 50 epochs)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('CGAN - Analise de Treino', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(run_dir / 'cgan_training_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- AMOSTRAS GERADAS (inline) ---
best_model = CGANGenerator(
    latent_dim=config.latent_dim, embed_dim=config.embed_dim,
    num_classes=10, base_ch=config.base_channels,
).to(device)
best_model.load_state_dict(_load_torch_state_dict(best_ckpt, device))
best_model.eval()

class_names = ['art_nouveau', 'baroque', 'expressionism', 'impressionism', 'post_impressionism',
               'realism', 'renaissance', 'romanticism', 'surrealism', 'ukiyo_e']

fig, axes = plt.subplots(10, 8, figsize=(16, 20))
with torch.no_grad():
    for cls in range(10):
        labels = torch.full((8,), cls, device=device, dtype=torch.long)
        rng = _make_rng(20260625 + cls)
        z = torch.randn(8, config.latent_dim, 1, 1, generator=rng, device=device)
        imgs = denormalize(best_model(z, labels)).cpu()
        for j in range(8):
            axes[cls, j].imshow(imgs[j].permute(1, 2, 0).numpy())
            axes[cls, j].axis('off')
        axes[cls, 0].set_ylabel(class_names[cls], fontsize=8, rotation=0, labelpad=80)

plt.suptitle('CGAN - Amostras Geradas por Classe (8 por classe, best checkpoint)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(str(run_dir / 'cgan_samples_per_class.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\nMelhor FID durante treino: {best_fid:.2f}')
print(f'Modelo guardado em: {run_dir}')

## 7. Avaliacao Final CGAN (10 seeds x 5000 amostras)

A celula seguinte carrega o melhor checkpoint e corre a avaliacao rigorosa. **Pode ser re-executada independentemente** sem repetir o treino.

**Resultado: FID = 99.65 +/- 1.26 | KID = 0.0744 +/- 0.0012**

In [ ]:
# ============================================================
# AVALIACAO FINAL CGAN (10 seeds x 5000 amostras)
# Pode ser re-executada independentemente do treino
# ============================================================
import json
import numpy as np


def _load_torch_state_dict(path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _make_rng(seed):
    try:
        rng = torch.Generator(device=device)
    except (TypeError, RuntimeError):
        rng = torch.Generator()
    rng.manual_seed(int(seed))
    return rng


run_dir = OUTPUT_ROOT / 'cgan_final_full_artbench10'
config = TrainConfig(
    base_channels=128, latent_dim=100, embed_dim=50,
    epochs=400, lr_g=1.5e-4, lr_d=1e-4, n_critic=2, label_smooth=0.9,
)
history_file = run_dir / 'cgan_history.json'
eval_file = run_dir / 'cgan_eval_results.json'
best_ckpt = run_dir / 'cgan_generator_best.pt'

if not best_ckpt.exists():
    raise FileNotFoundError(f'Checkpoint final nao encontrado: {best_ckpt}')

# --- 1. GRAFICOS DE TREINO ---
if history_file.exists():
    history = json.loads(history_file.read_text('utf-8'))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history['d_loss'], label='D Loss', color='#e74c3c', alpha=0.8)
    axes[0].plot(history['g_loss'], label='G Loss', color='#3498db', alpha=0.8)
    axes[0].axhline(0.693, color='black', linestyle='--', alpha=0.5, label='Nash (0.693)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('CGAN Training Loss (400 epochs)')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    fid_data = history.get('fid_quick', [])
    if fid_data:
        if isinstance(fid_data[0], dict):
            fid_epochs = [x['epoch'] for x in fid_data]
            fid_vals = [x['fid'] for x in fid_data]
        else:
            fid_epochs = list(range(10, 10 * len(fid_data) + 1, 10))
            fid_vals = fid_data
        best_idx = fid_vals.index(min(fid_vals))
        axes[1].plot(fid_epochs, fid_vals, 'o-', color='#9b59b6', markersize=4)
        axes[1].scatter([fid_epochs[best_idx]], [fid_vals[best_idx]], color='red', s=100, zorder=5,
                        label=f'Best: {min(fid_vals):.2f} (ep {fid_epochs[best_idx]})')
        axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('FID (2000 samples)')
        axes[1].set_title('Quick-FID Evolution')
        axes[1].legend(); axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, 'Sem dados de FID', ha='center', va='center')
        axes[1].set_axis_off()

    axes[2].hist(history['d_loss'][-50:], bins=20, color='#e74c3c', alpha=0.7, label='D Loss (last 50)')
    axes[2].hist(history['g_loss'][-50:], bins=20, color='#3498db', alpha=0.7, label='G Loss (last 50)')
    axes[2].axvline(0.693, color='black', linestyle='--', label='Nash')
    axes[2].set_title('Loss Distribution (last 50 epochs)')
    axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle('CGAN - Analise de Treino', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig(str(run_dir / 'cgan_training_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print(f'Historico de treino nao encontrado: {history_file}')

# --- 2. AMOSTRAS GERADAS (Melhor Checkpoint) ---
print('\n=== AMOSTRAS GERADAS (Melhor Checkpoint) ===')
best_G = CGANGenerator(
    latent_dim=config.latent_dim, embed_dim=config.embed_dim,
    num_classes=10, base_ch=config.base_channels,
).to(device)
best_G.load_state_dict(_load_torch_state_dict(best_ckpt, device))
best_G.eval()

class_names = ['art_nouveau', 'baroque', 'expressionism', 'impressionism', 'post_impressionism',
               'realism', 'renaissance', 'romanticism', 'surrealism', 'ukiyo_e']

fig, axes = plt.subplots(10, 8, figsize=(16, 20))
with torch.no_grad():
    for cls in range(10):
        labels = torch.full((8,), cls, device=device, dtype=torch.long)
        rng = _make_rng(20260725 + cls)
        z = torch.randn(8, config.latent_dim, 1, 1, generator=rng, device=device)
        imgs = denormalize(best_G(z, labels)).cpu()
        for j in range(8):
            axes[cls, j].imshow(imgs[j].permute(1, 2, 0).numpy())
            axes[cls, j].axis('off')
        axes[cls, 0].set_ylabel(class_names[cls], fontsize=8, rotation=0, labelpad=80)

plt.suptitle('CGAN - Amostras Geradas por Classe (8 por classe, best checkpoint)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(str(run_dir / 'cgan_samples_per_class.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 3. AVALIACAO RIGOROSA (10 seeds x 5000 amostras) ---
if eval_file.exists():
    final_results = json.loads(eval_file.read_text('utf-8'))
    print('\nResultados carregados de avaliacao existente.')
else:
    print('\n=== AVALIACAO RIGOROSA CGAN (10 seeds x 5000 amostras) ===')
    _, eval_loader = build_loaders(device_batch_size=128, seed=42, use_full_data=True)
    final_results = evaluate_final(
        run_dir=run_dir, eval_loader=eval_loader,
        config=config, device=device, repeats=10, n_gen=5000,
    )

# --- 4. Graficos de avaliacao ---
fid_arr = np.array(final_results['fid'])
kid_arr = np.array(final_results['kid'])
if len(fid_arr) == 0 or len(kid_arr) == 0:
    raise ValueError('Resultados de avaliacao vazios; nao e possivel desenhar graficos.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(range(len(fid_arr)), fid_arr, 'o-', color='#9b59b6', linewidth=2, markersize=6)
axes[0].axhline(fid_arr.mean(), color='red', linestyle='--',
                label=f'Mean: {fid_arr.mean():.2f} +/- {fid_arr.std():.2f}')
axes[0].set_xlabel('Seed'); axes[0].set_ylabel('FID')
axes[0].set_title('FID per Seed (5000 samples)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(range(len(kid_arr)), kid_arr, 'o-', color='#e67e22', linewidth=2, markersize=6)
axes[1].axhline(kid_arr.mean(), color='red', linestyle='--',
                label=f'Mean: {kid_arr.mean():.4f} +/- {kid_arr.std():.4f}')
axes[1].set_xlabel('Seed'); axes[1].set_ylabel('KID')
axes[1].set_title('KID per Seed (5000 samples)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('CGAN - Avaliacao Rigorosa', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(str(run_dir / 'cgan_eval_per_seed.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- 5. RESULTADO FINAL ---
print(f'\n{"="*50}')
print('  CGAN RESULTADO FINAL')
print(f'  FID: {fid_arr.mean():.2f} +/- {fid_arr.std():.2f}')
print(f'  KID: {kid_arr.mean():.4f} +/- {kid_arr.std():.4f}')
print(f'{"="*50}')

## Notas Finais

O CGAN mostra duas coisas ao mesmo tempo:

1. **Condicionamento por classe funciona** - o modelo gera imagens condicionadas a cada uma das 10 classes artisticas.
2. **O custo da condicionalidade continua visivel** - o CGAN melhora claramente face aos autoencoders, mas fica atras do DCGAN em qualidade global.

| Modelo | FID | KID |
|--------|-----|-----|
| DCGAN | 25.39 +/- 1.53 | 0.0118 +/- 0.0004 |
| CGAN | 99.65 +/- 1.26 | 0.0744 +/- 0.0012 |

O DCGAN supera o CGAN em fidelidade global, mas o CGAN oferece controlo sobre o estilo gerado, uma propriedade importante para aplicacoes praticas de arte generativa.